In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave

np.random.seed(123)

In [10]:
def inv_pareto(u, th):
    return (1 - u)**(1 / (1 - th))

def med(th):
    return 2 ** (1/(th-1))

n = 100
conf = 0.95

th_true = 5
x = inv_pareto(st.uniform(loc=0, scale=1).rvs(size=n), th=th_true)
x

array([1.34725345, 1.08791919, 1.06643469, 1.22184061, 1.37405708,
       1.14742968, 2.68517396, 1.33463863, 1.1781319 , 1.13251795,
       1.11080565, 1.38604584, 1.15525147, 1.01550213, 1.13529535,
       1.39772835, 1.05166389, 1.04941189, 1.20874449, 1.20892274,
       1.28602301, 1.60533872, 1.38023169, 1.26624914, 1.37772362,
       1.10241787, 1.11881652, 1.06692208, 1.09082443, 1.28302874,
       1.02445075, 1.15275917, 1.15131921, 1.18548161, 1.14878811,
       1.09810564, 1.14904886, 1.7500468 , 2.05713921, 1.19030172,
       1.27699576, 1.03119328, 1.10012039, 1.14334891, 1.653768  ,
       1.0747331 , 1.17932787, 2.88473942, 1.20108375, 1.26777643,
       1.03265898, 1.54908608, 1.25984994, 1.21762451, 1.1106306 ,
       1.09488003, 1.14442411, 1.33092856, 1.68333298, 1.19548629,
       1.31869991, 1.24661766, 1.27780401, 1.32411385, 1.58697943,
       1.02195261, 1.43425438, 1.07231319, 1.05547093, 1.23667346,
       1.02547097, 1.71844067, 1.27980938, 1.37893367, 1.00407

In [11]:
th_hat = 1 + 1 / (np.mean(np.log(x)))
med_true = med(th_true)
med_hat = med(th_hat)
z_low = st.norm.ppf((1-conf)/2)
z_high = st.norm.ppf((1+conf)/2)
ci_asym_med_left = med_hat - med_hat*np.log(2)/(np.sqrt(n)*(th_hat-1))*z_high
ci_asym_med_right = med_hat - med_hat*np.log(2)/(np.sqrt(n)*(th_hat-1))*z_low
ci_asym_med_len = ci_asym_med_right - ci_asym_med_left
print(f"Медиана: {med_true}, оценка медианы:{med_hat}")
print("Асимптотический доверительный интервал для медианы:", ci_asym_med_left, "< theta <", ci_asym_med_right)
print("length =", ci_asym_med_len)

Медиана: 1.189207115002721, оценка медианы:1.1724303006784136
Асимптотический доверительный интервал для медианы: 1.135875252411483 < theta < 1.2089853489453442
length = 0.07311009653386114


In [12]:
z_low = st.norm.ppf((1-conf)/2)
z_high = st.norm.ppf((1+conf)/2)
ci_asym_left = th_hat - z_high / (np.sum(np.log(x))/np.sqrt(n))
ci_asym_right = th_hat - z_low / (np.sum(np.log(x))/np.sqrt(n))
ci_asym_len = ci_asym_right - ci_asym_left
print(f"theta: {th_true}, MLE theta:{th_hat}")
print("Асимптотический доверительный интервал для theta:", ci_asym_left, "< theta <", ci_asym_right)
print("length =", ci_asym_len)

theta: 5, MLE theta:5.357257482320771
Асимптотический доверительный интервал для theta: 4.503250708649133 < theta < 6.2112642559924085
length = 1.7080135473432758


In [13]:
B = 1000

def boot_np(samp, B):
    diffs = []
    m = len(samp)
    for _ in range(B):
        samp_b = np.random.choice(samp, size=m, replace=True)        
        diffs += [(1 + 1 / (np.mean(np.log(samp_b)))) - th_true]
    return sorted(np.array(diffs))

diffs_np = boot_np(x, B)
idx_low = int((1 - conf)/2 * B - 1)
idx_high = int((1 + conf)/2 * B - 1)

ci_np_left = th_hat - diffs_np[idx_high]
ci_np_right = th_hat - diffs_np[idx_low]
ci_np_len = ci_np_right - ci_np_left
print("Непараметрический бутстраповский доверительный интервал для theta(ОМП):", ci_np_left, "< theta <", ci_np_right)
print("length =", ci_np_len)

Непараметрический бутстраповский доверительный интервал для theta(ОМП): 4.021446553471507 < theta < 5.7433252383298115
length = 1.7218786848583045


In [ ]:
B = 50000

def boot_p(samp, B):
    diffs = []
    m = len(samp)
    for _ in range(B):
        samp_b = inv_pareto(st.uniform(loc=0, scale=1).rvs(size=m), th=th_hat)        
        diffs += [(1 + 1 / (np.mean(np.log(samp_b)))) - th_true]
    return sorted(np.array(diffs))

diffs_p = boot_p(x, B)
idx_low = int((1 - conf)/2 * B - 1)
idx_high = int((1 + conf)/2 * B - 1)
ci_p_left = th_hat - diffs_p[idx_high]
ci_p_right = th_hat - diffs_p[idx_low]
ci_p_len = ci_p_right - ci_p_left
print("Параметрический бутстраповский доверительный интервал для theta(ОМП):", ci_p_left, "< theta <", ci_p_right)
print("length =", ci_p_len)

Параметрический бутстраповский доверительный интервал для theta(ОМП): 4.094009532172082 < theta < 5.712518699994053
length = 1.618509167821971


In [15]:
lens = sorted([(ci_asym_len, "Асимптотический"), 
               (ci_np_len, "Бутстрап непараметрический"), 
               (ci_p_len, "Бутстрап параметрический")])
print("Рейтинг (для theta):")
for i in range(len(lens)):
    print(f"{i+1})", lens[i][1], f"(len = {np.round(lens[i][0],3)})")

Рейтинг (для theta):
1) Бутстрап параметрический (len = 1.619)
2) Асимптотический (len = 1.708)
3) Бутстрап непараметрический (len = 1.722)
